In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
!nvidia-smi

Sun Mar 29 16:12:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L40S                    On  |   00000000:4A:00.0 Off |                    0 |
| N/A   28C    P8             33W /  350W |       0MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
# Put project root on sys.path so "source" is importable
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent  # notebooks/ -> root/
print(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

/home/mpominova/lcms-foundation-model


In [ ]:
import os
import yaml
import numpy as np
import pandas as pd
import polars as pl

import torch
import torch.nn as nn
import pytorch_lightning as L
from tqdm import tqdm
from depthcharge.data import SpectrumDataset, spectra_to_df, preprocessing
from torch.utils.data import DataLoader

from source.dataset import LanceMapDataset, RunDataset
from source.model import MS1Encoder
from source.scheduler import CosineWarmupScheduler
from source.config import ExperimentConfig, DataConfig, ModelConfig, OptimizerConfig, TrainingConfig

/home/mpominova/.local/share/mamba/envs/lcms-foundation/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# last version of MS1Encoder model source code # 12/03/26 

import numpy as np
import pandas as pd # DEBUG
import torch
import torch.nn as nn
import torchmetrics
import pytorch_lightning as L
from depthcharge.encoders import PeakEncoder, PositionalEncoder
from depthcharge.transformers import SpectrumTransformerEncoder

from IPython.display import clear_output # DEBUG
pd.set_option('display.max_rows', 500) # DEBUG

class MS1Encoder(L.LightningModule):
    def __init__(
        self,
        d_model=128,
        nhead=8,
        dim_feedforward=512,
        n_layers=4,
        dropout=0.1,
        n_bins=2000,
        bin_mz_min=0,
        bin_mz_max=2000,
        masked_peaks_fraction=0.3,
        mask_proportional=True,
        lr=5e-4,
        warmup_iters=1000,
        cosine_schedule_period_iters=32000,
    ):
        super().__init__()
        self.save_hyperparameters()

        self.d_model = d_model
        self.nhead = nhead
        self.dim_feedforward = dim_feedforward
        self.n_layers = n_layers
        self.dropout = dropout

        self.n_bins = n_bins
        self.bin_mz_min = bin_mz_min
        self.bin_mz_max = bin_mz_max
        self.masked_peaks_fraction = masked_peaks_fraction
        self.mask_proportional = mask_proportional

        self.peak_encoder = nn.Sequential(
            PeakEncoder(
                d_model=self.d_model,
                min_mz_wavelength=0.001,
                max_mz_wavelength=10000,
                min_intensity_wavelength=1e-06,
                max_intensity_wavelength=1,
                learnable_wavelengths=False,
            ),
        )
        self.encoder = SpectrumTransformerEncoder(
            d_model=self.d_model,
            nhead=self.nhead,
            dim_feedforward=self.dim_feedforward,  # 1024,
            n_layers=self.n_layers,
            dropout=self.dropout,
            peak_encoder=self.peak_encoder,
        )

        self.head_mz = nn.Sequential(
            nn.Linear(d_model, n_bins),
        )  # outputs n_bins logits for each peak
        self.head_I = nn.Sequential(
            nn.Linear(d_model, 1),
        )  # outputs float I value for each peak

        # losses
        self.loss_mz_bin = nn.CrossEntropyLoss(reduction="mean", ignore_index=-1)
        self.loss_I = nn.MSELoss(reduction="mean")
        # metrics
        self.train_accuracy_mz_bin = torchmetrics.classification.Accuracy(task="multiclass", num_classes=self.n_bins, ignore_index=-1)
        self.val_accuracy_mz_bin = torchmetrics.classification.Accuracy(task="multiclass", num_classes=self.n_bins, ignore_index=-1)

    def get_peaks_mask(self, intensities, proportional=False):
        if proportional:
            k = int(intensities.size(1) * self.masked_peaks_fraction)
            mask = torch.zeros_like(intensities, dtype=torch.bool)
            # FIXME: assume we never have zero rows (= spectra with no peaks), otherwise will fail

            # compute sampling weights w
            I_mean = intensities.sum(dim=1) / (intensities != 0).sum(dim=1) # mean I of non-zero peaks
            w = intensities + (intensities == 0).float() * I_mean[:, None]  # weight 0s by mean I
            # sample k indices without replacement, weighted by w
            idx = torch.multinomial(w, num_samples=k, replacement=False)
            mask = mask.scatter(dim=1, index=idx.to(dtype=torch.int64), value=True) # value to write into mask (True)

        else:
            mask = torch.rand_like(intensities) < self.masked_peaks_fraction
        return mask

    def get_mz_bins(self, mz):
        # every peak with mz > bin_mz_max will belong to max bin
        mz = mz.clamp(0, self.bin_mz_max - 1)
        mz_binned = (
            ((mz - self.bin_mz_min) / (self.bin_mz_max - self.bin_mz_min) * self.n_bins)
            .floor()
            .long()
        )
        mz_binned[mz < self.bin_mz_min] = -1
        return mz_binned

    def forward(
        self,
        mzs: torch.Tensor,
        intensities: torch.Tensor,
    ):
        peak_embs, _ = self.encoder(mz_array=mzs, intensity_array=intensities)
        # drop global token
        peak_embs = peak_embs[:, 1:, :]
        return peak_embs

    # def ssl_step(self):
    #     # TODO: move here the repeated part of training & validation parts
    #     return

    def training_step(self, batch, batch_idx):
        mz = batch["mz_array"]
        I = batch["intensity_array"]

        # sample peak masks
        masks = self.get_peaks_mask(I, proportional=self.mask_proportional)

        # prepare targets (bins & I of masked peaks)
        target_mz, target_I = mz[masks], I[masks]
        # transform mz into bins (target classes C \in [0, n_bins - 1])
        target_mz_bins = self.get_mz_bins(target_mz)

        # mask input peaks with 0 (before encoding)
        masked_mz = mz * (1 - masks.float())
        # masked_I = I * (1 - masks.float()) # FIX: not mask intensities, only mz
        
        # get embeddings for all peaks
        # peak_embs = self.forward(masked_mz, masked_I)
        peak_embs = self.forward(masked_mz, I) # FIX: not mask intensities, only mz
        # select only embeddings of masked peaks
        masked_peak_embs = peak_embs[masks]
        # predict masked peaks binned mz & I
        pred_mz_bins = self.head_mz(masked_peak_embs)
        pred_I = self.head_I(masked_peak_embs).squeeze(dim=-1)

        loss_mz_bin = self.loss_mz_bin(pred_mz_bins, target_mz_bins)
        loss_I = self.loss_I(pred_I, target_I)
        loss = loss_mz_bin# + loss_I
        self.log("train_loss_mz_bin", loss_mz_bin.item())
        # self.log("train_loss_I", loss_I.item())
        self.log("train_loss", loss.item())
        # Accuracy metric for mz bin prediction
        acc_mz_bin = self.train_accuracy_mz_bin(pred_mz_bins, target_mz_bins)
        self.log("train_acc_mz_bin", acc_mz_bin.item(), prog_bar=True, on_step=True, on_epoch=False)
        return loss

    def validation_step(self, batch, batch_idx):
        # Note: validation is now always partially random (mask sampling)
        # so not identical between epochs. May want to change it (how?).
        mz = batch["mz_array"]
        I = batch["intensity_array"]

        # sample peak masks
        masks = self.get_peaks_mask(I, proportional=self.mask_proportional)

        # prepare targets (bins & I of masked peaks)
        target_mz, target_I = mz[masks], I[masks]
        # transform mz into bins (target classes C \in [0, n_bins - 1])
        target_mz_bins = self.get_mz_bins(target_mz)

        # mask input peaks with 0 (before encoding)
        masked_mz = mz * (1 - masks.float())
        # masked_I = I * (1 - masks.float()) # FIX: not mask intensities, only mz
        
        # get embeddings for all peaks
        # peak_embs = self.forward(masked_mz, masked_I)
        peak_embs = self.forward(masked_mz, I) # FIX: not mask intensities, only mz
        # select only embeddings of masked peaks
        masked_peak_embs = peak_embs[masks]
        # predict masked peaks binned mz & I
        pred_mz_bins = self.head_mz(masked_peak_embs)
        pred_I = self.head_I(masked_peak_embs).squeeze(dim=-1)

        loss_mz_bin = self.loss_mz_bin(pred_mz_bins, target_mz_bins)
        # loss_I = self.loss_I(pred_I, target_I)
        loss = loss_mz_bin# + loss_I
        self.log("val_loss_mz_bin", loss_mz_bin.item())
        # self.log("val_loss_I", loss_I.item())
        self.log("val_loss", loss.item())
        # Accuracy metric for mz bin prediction
        acc_mz_bin = self.val_accuracy_mz_bin(pred_mz_bins, target_mz_bins)
        self.log("val_acc_mz_bin", acc_mz_bin.item(), prog_bar=True, on_step=False, on_epoch=True)

        # # Display prediction sample
        # n = 30
        # mz_bins_true, I_true = target_mz_bins[:n].cpu().numpy(), target_I[:n].cpu().numpy()
        # mz_bins_pred, I_pred = pred_mz_bins[:n].argmax(dim=1).cpu().numpy(), pred_I[:n].cpu().numpy()
        # sample_df = np.column_stack((
        #     mz_bins_true.ravel(), 
        #     I_true.ravel(), 
        #     mz_bins_pred.ravel(), 
        #     # I_pred.ravel()
        # ))
        # sample_df = pd.DataFrame(
        #     sample_df, columns=[
        #         "mz_bins_true", 
        #         "I_true", 
        #         "mz_bins_pred", 
        #         # "I_pred"
        #     ]
        # )
        # display(sample_df)
        
        return loss

    def on_validation_epoch_start(self,):
        clear_output(True)
        return

    def configure_optimizers(
        self,
    ):
        """TODO."""
        optimizer = torch.optim.Adam(
            self.parameters(), lr=self.hparams.lr, betas=(0.9, 0.98)
        )
        self.lr_scheduler = CosineWarmupScheduler(
            optimizer,
            self.hparams.warmup_iters,
            self.hparams.cosine_schedule_period_iters,
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": self.lr_scheduler,
                "interval": "step",
                "frequency": 1,
                "name": "cosine_warmup",
            },
        }

    def optimizer_step(self, *args, **kwargs):
        super().optimizer_step(*args, **kwargs)
        self.log("lr", self.lr_scheduler.get_last_lr()[0])

In [7]:
def load_config(config_path):
    """Load configuration from YAML file."""
    with open(config_path, "r") as f:
        config_dict = yaml.safe_load(f)
        config = ExperimentConfig(
            name=config_dict['name'],
            data=DataConfig(**config_dict['data']),
            model=ModelConfig(**config_dict['model']),
            optimizer=OptimizerConfig(**config_dict['optimizer']),
            training=TrainingConfig(**config_dict['training'])
        )
    return config

In [8]:
# Load training data
data_root_dir = "/mnt/data/shared/lc_ms_foundation/"
dset_name = "abele_data"
data_dir = os.path.join(data_root_dir, dset_name, "mzml")

mzml_files = os.listdir(data_dir)
print("Total N files:", len(mzml_files))

Total N files: 1334


In [9]:
selected_genuses = [
    "Pseudomonas", 
    "Staphylococcus", 
    "Bacillus", 
    "Escherichia", 
    "Enterococcus",
    "Serratia",
    "Acinetobacter",
]

In [10]:
meta_df = pl.read_csv(os.path.join(
    data_root_dir, 
    dset_name,
    "all_abele_metadata.csv"
))

meta_df = meta_df.rename({
    "characteristics[organism]": "organism",
    "comment[data file]": "data_file"
})

meta_df = meta_df.filter(meta_df["genus"].is_in(selected_genuses))
meta_df

organism,genus,data_file
str,str,str
"""Bacillus cereus""","""Bacillus""","""BBM_429_P110_31_MIA_007"""
"""Pseudomonas aeruginosa""","""Pseudomonas""","""BBM_437_P110_31_MIA_036"""
"""Bacillus pumilus""","""Bacillus""","""BBM_429_P110_31_MIA_023"""
"""Serratia fonticola""","""Serratia""","""BBM_437_P110_31_MIA_040"""
"""Staphylococcus nepalensis""","""Staphylococcus""","""BBM_441_P110_31_MIA_001"""
…,…,…
"""Escherichia coli""","""Escherichia""","""BBM_749_P110_38_MIA_095"""
"""Escherichia coli""","""Escherichia""","""BBM_749_P110_38_MIA_096"""
"""Pseudomonas aeruginosa""","""Pseudomonas""","""BBM_750_P110_38_MIA_001"""


In [11]:
meta_df["genus"].value_counts()

genus,count
str,u32
"""Staphylococcus""",136
"""Acinetobacter""",24
"""Serratia""",27
"""Escherichia""",51
"""Enterococcus""",42
"""Bacillus""",109
"""Pseudomonas""",312


In [ ]:
# Pseudomonas, Staphylococcus, Bacillus, Escherichia are used for petraining
# and Enterococcus and Serratia are split between train and val for downstream eval
# - group by organism, put all files of the same organism in one split

In [13]:
# Use our "default" parameters from config to load data

# TODO: do we maybe need separate configs for different types of experiments?
# (e.g. MS2, MS1, online eval, retrain eval, run, ... ?)
# TODO: would be nice to fix some convenient naming convention for experiments

config = load_config("../config.yaml")
config

ExperimentConfig(name='ms1-mz-200_peaks-sqrt-clf_run_retrain', data=DataConfig(train_dir='train_mzml', val_dir='val_mzml', batch_size=2000, max_num_peaks=200), model=ModelConfig(d_model=256, nhead=8, dim_feedforward=512, n_layers=6, dropout=0.1, n_bins=1200, bin_mz_min=300, bin_mz_max=1500, masked_peaks_fraction=0.3), optimizer=OptimizerConfig(lr=0.0005, warmup_iters=1000, cosine_schedule_period_iters=64000), training=TrainingConfig(checkpoint_path='./train_checkpoints', max_epochs=1000, gradient_clip_val=5, accelerator='gpu', devices=1))

In [14]:
data_dir = "/mnt/data/mpominova/lcms_foundation_data/abele_data"
len(os.listdir(data_dir)), os.listdir(data_dir)[:2]

(701, ['BBM_429_P110_31_MIA_007.parquet', 'BBM_437_P110_31_MIA_036.parquet'])

In [15]:
dfs = {
    mzml_file: pl.read_parquet(os.path.join(data_dir, mzml_file))
    for mzml_file in tqdm(os.listdir(data_dir))
}
len(dfs)

100%|██████████| 701/701 [00:26<00:00, 26.23it/s]


701

In [16]:
meta_df["genus"].value_counts()

genus,count
str,u32
"""Pseudomonas""",312
"""Staphylococcus""",136
"""Serratia""",27
"""Enterococcus""",42
"""Bacillus""",109
"""Acinetobacter""",24
"""Escherichia""",51


In [18]:
genus_class = {
    "Enterococcus": 0,
    "Serratia": 1,
    "Acinetobacter": 2,
    #
    "Pseudomonas": 3,
    "Staphylococcus": 4,
    "Bacillus": 5,
    "Escherichia": 6,
}
meta_df = meta_df.with_columns((pl.col("data_file") + ".mzML").alias("peak_file"))
meta_df = meta_df.with_columns(
    (pl.col("genus").replace(genus_class, return_dtype=int).alias("genus_class"))
)

display(meta_df["genus_class"].value_counts())
meta_df

/tmp/ipykernel_23902/3550003717.py:13: DeprecationWarning: the `return_dtype` parameter for `replace` is deprecated. Use `replace_strict` instead to set a return data type while replacing values.
(Deprecated in version 1.0.0)
  (pl.col("genus").replace(genus_class, return_dtype=int).alias("genus_class"))


genus_class,count
i64,u32
2,24
6,51
1,27
4,136
3,312
0,42
5,109


organism,genus,data_file,peak_file,genus_class
str,str,str,str,i64
"""Bacillus cereus""","""Bacillus""","""BBM_429_P110_31_MIA_007""","""BBM_429_P110_31_MIA_007.mzML""",5
"""Pseudomonas aeruginosa""","""Pseudomonas""","""BBM_437_P110_31_MIA_036""","""BBM_437_P110_31_MIA_036.mzML""",3
"""Bacillus pumilus""","""Bacillus""","""BBM_429_P110_31_MIA_023""","""BBM_429_P110_31_MIA_023.mzML""",5
"""Serratia fonticola""","""Serratia""","""BBM_437_P110_31_MIA_040""","""BBM_437_P110_31_MIA_040.mzML""",1
"""Staphylococcus nepalensis""","""Staphylococcus""","""BBM_441_P110_31_MIA_001""","""BBM_441_P110_31_MIA_001.mzML""",4
…,…,…,…,…
"""Escherichia coli""","""Escherichia""","""BBM_749_P110_38_MIA_095""","""BBM_749_P110_38_MIA_095.mzML""",6
"""Escherichia coli""","""Escherichia""","""BBM_749_P110_38_MIA_096""","""BBM_749_P110_38_MIA_096.mzML""",6
"""Pseudomonas aeruginosa""","""Pseudomonas""","""BBM_750_P110_38_MIA_001""","""BBM_750_P110_38_MIA_001.mzML""",3


In [19]:
train_genuses = ['Pseudomonas', 'Staphylococcus', 'Bacillus', 'Escherichia']
probe_genuses = ["Enterococcus", "Serratia"]

In [20]:
g = meta_df.select(["organism", "genus_class"]).group_by("organism")
sorted_organisms = g.first().sort(by=["genus_class", "organism"])["genus_class", "organism"]
sorted_organisms

genus_class,organism
i64,str
0,"""Enterococcus avium"""
0,"""Enterococcus casseliflavus"""
0,"""Enterococcus cecorum"""
0,"""Enterococcus columbae"""
0,"""Enterococcus dispar"""
…,…
5,"""Bacillus sonorensis"""
5,"""Bacillus subtilis"""
5,"""Bacillus thuringiensis"""


In [ ]:
train_organisms = sorted_organisms.filter(
    sorted_organisms["genus_class"] > 2
)["organism"].to_list()

val_organisms = sorted_organisms.filter(
    sorted_organisms["genus_class"].is_in([0, 1])
)["organism"].to_list()

probe_train_organisms = val_organisms[::2]
probe_val_organisms = [
    organism for organism in val_organisms 
    if organism not in probe_train_organisms
]

len(train_organisms), len(probe_train_organisms), len(probe_val_organisms)

(82, 12, 11)

In [22]:
probe_val_organisms

['Enterococcus casseliflavus',
 'Enterococcus columbae',
 'Enterococcus durans',
 'Enterococcus faecium',
 'Enterococcus malodoratus',
 'Enterococcus raffinosus',
 'Enterococcus sulfureus',
 'Serratia fonticola',
 'Serratia liquefaciens',
 'Serratia odorifera',
 'Serratia proteamaculans']

In [23]:
display(sorted_organisms.filter(
    sorted_organisms["organism"].is_in(probe_train_organisms)
))
display(sorted_organisms.filter(
    sorted_organisms["organism"].is_in(probe_val_organisms)
))

genus_class,organism
i64,str
0,"""Enterococcus avium"""
0,"""Enterococcus cecorum"""
0,"""Enterococcus dispar"""
0,"""Enterococcus faecalis"""
0,"""Enterococcus hirae"""
…,…
1,"""Serratia ficaria"""
1,"""Serratia grimesii"""
1,"""Serratia marcescens"""


genus_class,organism
i64,str
0,"""Enterococcus casseliflavus"""
0,"""Enterococcus columbae"""
0,"""Enterococcus durans"""
0,"""Enterococcus faecium"""
0,"""Enterococcus malodoratus"""
…,…
0,"""Enterococcus sulfureus"""
1,"""Serratia fonticola"""
1,"""Serratia liquefaciens"""


In [24]:
organism2split = {}
for organism in train_organisms:
    organism2split[organism] = "train"
for organism in probe_train_organisms:
    organism2split[organism] = "probe_train"
for organism in probe_val_organisms:
    organism2split[organism] = "probe_val"
    
meta_df = meta_df.with_columns(
    pl.col("organism").replace(organism2split).alias("split")
)
meta_df

organism,genus,data_file,peak_file,genus_class,split
str,str,str,str,i64,str
"""Bacillus cereus""","""Bacillus""","""BBM_429_P110_31_MIA_007""","""BBM_429_P110_31_MIA_007.mzML""",5,"""train"""
"""Pseudomonas aeruginosa""","""Pseudomonas""","""BBM_437_P110_31_MIA_036""","""BBM_437_P110_31_MIA_036.mzML""",3,"""train"""
"""Bacillus pumilus""","""Bacillus""","""BBM_429_P110_31_MIA_023""","""BBM_429_P110_31_MIA_023.mzML""",5,"""train"""
"""Serratia fonticola""","""Serratia""","""BBM_437_P110_31_MIA_040""","""BBM_437_P110_31_MIA_040.mzML""",1,"""probe_val"""
"""Staphylococcus nepalensis""","""Staphylococcus""","""BBM_441_P110_31_MIA_001""","""BBM_441_P110_31_MIA_001.mzML""",4,"""train"""
…,…,…,…,…,…
"""Escherichia coli""","""Escherichia""","""BBM_749_P110_38_MIA_095""","""BBM_749_P110_38_MIA_095.mzML""",6,"""train"""
"""Escherichia coli""","""Escherichia""","""BBM_749_P110_38_MIA_096""","""BBM_749_P110_38_MIA_096.mzML""",6,"""train"""
"""Pseudomonas aeruginosa""","""Pseudomonas""","""BBM_750_P110_38_MIA_001""","""BBM_750_P110_38_MIA_001.mzML""",3,"""train"""


In [25]:
# Data for SSL training
split_df = meta_df.filter(pl.col("split") == "train")
print("SSL train - Total:", len(split_df))
display(split_df["genus_class"].value_counts(normalize=False).sort(by="genus_class"))
# display(split_df["genus_class"].value_counts(normalize=True).sort(by="genus_class"))

# Data for downstream model training
split_df = meta_df.filter(pl.col("split") == "probe_train")
print("Downstream train - Total:", len(split_df))
display(split_df["genus_class"].value_counts(normalize=False).sort(by="genus_class"))
# display(split_df["genus_class"].value_counts(normalize=True).sort(by="genus_class"))

# Data for downstream model validation
split_df = meta_df.filter(pl.col("split") == "probe_val")
print("Downstream val - Total:", len(split_df))
display(split_df["genus_class"].value_counts(normalize=False).sort(by="genus_class"))
# display(split_df["genus_class"].value_counts(normalize=True).sort(by="genus_class"))

SSL train - Total: 608


genus_class,count
i64,u32
3,312
4,136
5,109
6,51


Downstream train - Total: 36


genus_class,count
i64,u32
0,21
1,15


Downstream val - Total: 33


genus_class,count
i64,u32
0,21
1,12


In [26]:
# train SpectrumDataset
train_df = pl.concat(
    [
        dfs[mzml_file.replace("mzML", "parquet")] for mzml_file 
        in meta_df.filter(pl.col("split") == "train")["peak_file"].to_list()
    ], 
    how="vertical"
)
train_df = train_df.join(meta_df, on="peak_file", how="left")

# val SpectrumDataset
val_df = pl.concat(
    [
        dfs[mzml_file.replace("mzML", "parquet")] for mzml_file 
        in meta_df.filter(
            pl.col("split").is_in(["probe_train", "probe_val"])
        )["peak_file"].to_list()
    ], 
    how="vertical"
)
val_df = val_df.join(meta_df, on="peak_file", how="left")

train_df.shape, val_df.shape

((1734898, 8), (215737, 8))

In [27]:
train_stream = SpectrumDataset(
    train_df.select(["mz_array", "intensity_array", "genus_class"]), 
    batch_size=256, # streaming batch size, not training batch size
)
print("N train spectra", train_stream.n_spectra)
train_lance_path = str(train_stream.path)   # this is the key
print("Lance dataset path:", train_lance_path)
train_dataset = LanceMapDataset(train_lance_path, seq_len=config.data.max_num_peaks)

val_stream = SpectrumDataset(
    val_df.select(["mz_array", "intensity_array", "genus_class"]), 
    batch_size=256, # streaming batch size, not training batch size
)
print("N train spectra", val_stream.n_spectra)
val_lance_path = str(val_stream.path)   # this is the key
print("Lance dataset path:", val_lance_path)
val_dataset = LanceMapDataset(val_lance_path, seq_len=config.data.max_num_peaks)

N train spectra 1734898
Lance dataset path: /tmp/tmpeoyazgo4/b66116b7-4e53-4e9a-a45d-7fc7bf3ae0fd.lance
N train spectra 215737
Lance dataset path: /tmp/tmpxx17rusb/a08482fb-d06c-40b5-9fb2-4f5a7120fad1.lance


In [28]:
run_labels = dict(zip(meta_df["peak_file"], meta_df["genus_class"]))

probe_train_dataset = RunDataset(
    [
        dfs[mzml_file.replace("mzML", "parquet")] for mzml_file 
        in meta_df.filter(pl.col("split") == "probe_train")["peak_file"].to_list()
    ], 
    run_labels=run_labels, 
    seq_len=config.data.max_num_peaks
)
probe_val_dataset = RunDataset(
    [
        dfs[mzml_file.replace("mzML", "parquet")] for mzml_file 
        in meta_df.filter(pl.col("split") == "probe_val")["peak_file"].to_list()
    ], 
    run_labels=run_labels,
    seq_len=config.data.max_num_peaks
)
len(probe_train_dataset), len(probe_val_dataset)

100%|██████████| 33/33 [00:00<00:00, 101.90it/s]


(36, 33)

In [29]:
def run_collate_fn(rows):
    # rows is a list of dicts, one per spectrum
    keys = rows[0].keys()
    batch = {}
    for key in keys:
        if key in ['mz_array', 'intensity_array']:
            batch[key] = [torch.tensor(r[key]) for r in rows]
        else:
            batch[key] = torch.tensor([r[key] for r in rows])
    return batch

In [30]:
BATCH_SIZE = config.data.batch_size

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    num_workers=0, 
    shuffle=True
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    num_workers=0, 
    shuffle=False
)
print("SSL:", len(train_loader), len(val_loader))

probe_train_loader = DataLoader(
    probe_train_dataset, 
    batch_size=BATCH_SIZE, 
    num_workers=0, 
    shuffle=True,
    collate_fn=run_collate_fn,
)
probe_val_loader = DataLoader(
    probe_val_dataset, 
    batch_size=BATCH_SIZE, 
    num_workers=0, 
    shuffle=False,
    collate_fn=run_collate_fn,
)
print("Downstream:", len(probe_train_loader), len(probe_val_loader))

SSL: 868 108
Downstream: 1 1


In [31]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from torchmetrics.functional import accuracy


# Full-retrain fine-tuner model
# Is applied at the end of every SSL validation epoch (Or every n epochs?)
# Every time before evaluation the model (needs to be with a fixed random seed?)
# is re-initialized and trained from scratch for probe_n_epochs (configured in init)


class FineTuner(L.Callback):
    def __init__(
        self,
        encoder_output_dim: int,
        num_classes: int,
        target_key: str,
        probe_train_loader,
        probe_val_loader, # =None,
        class_weights: Tensor | None = None,
        lr: float = 1e-2,
        n_epochs: int = 10,
        min_train_loss: float = 0,
    ) -> None:
        super().__init__()

        if class_weights is not None:
            assert class_weights.size(0) == self.num_classes, "number of class weights must equal `num_classes`"

        self.encoder_output_dim = encoder_output_dim
        self.num_classes = num_classes
        self.target_key = target_key
        self.class_weights = class_weights
        self.lr = lr
        self.n_epochs = n_epochs
        self.min_train_loss = min_train_loss

        self.probe_train_loader = probe_train_loader
        self.probe_val_loader = probe_val_loader

        self.optimizer: torch.optim.Optimizer

    def _init_model_opt(self, trainer: L.Trainer, pl_module: L.LightningModule) -> None:        
        # finetuner layer becomes a part of the main (embedding) model
        self.finetuner = nn.Linear(self.encoder_output_dim, self.num_classes).to(pl_module.device)
        # init optimizer inside the callback
        self.optimizer = torch.optim.Adam(self.finetuner.parameters(), lr=self.lr)

    def _encode_run(self, pl_module, run_mz, run_I):
        run_mz, run_I = run_mz.to(pl_module.device), run_I.to(pl_module.device)
        with torch.no_grad():
            peak_embs = pl_module.forward(run_mz, run_I)
            spec_embs = peak_embs.mean(dim=1) # average peak embeddings
            spec_embs = spec_embs.unsqueeze(dim=0) # (1, T, d)
            run_emb = spec_embs.mean(dim=1) # (1, d)
        return run_emb

    def _step(self, pl_module, batch, train=True):
        # extract components from batch
        runs_mz = batch["mz_array"]
        runs_I = batch["intensity_array"]
        targets = batch[self.target_key].to(pl_module.device)

        runs_emb = [
            self._encode_run(pl_module, runs_mz[i], runs_I[i])
            for i in range(len(runs_mz))
        ]
        embs = torch.cat(runs_emb, dim=0) 

        # apply fine-tuner model
        preds = self.finetuner(embs)

        # compute loss
        loss = F.cross_entropy(preds, targets, weight=self.class_weights)

        if train:
            loss.backward()
            self.optimizer.step()
            self.optimizer.zero_grad()
        
        acc = accuracy(preds, targets, task="multiclass", num_classes=self.num_classes)

        # Display prediction sample
        if not train:
            n = 30
            sample_df = np.hstack((
                targets[:n].cpu().numpy()[:, None],
                F.softmax(preds, dim=1)[:n].detach().cpu().numpy(),
            ))
            sample_df = pd.DataFrame(
                sample_df, columns=[
                    "targets", 
                    "prob_0", 
                    "prob_1", 
                    # "prob_2", 
                    # "prob_3", 
                ]
            )
            display(sample_df)
        return loss.detach(), acc.detach()

    def on_train_epoch_end(self, trainer: L.Trainer, pl_module: L.LightningModule) -> None:
        """
        Init the probe head and train for probe_n_epochs passes
        over probe_train_loader, after SSL epoch ends.
        """
        self._init_model_opt(trainer, pl_module)

        was_training = pl_module.training
        pl_module.eval()
        self.finetuner.train()

        # train model for self.n_epochs on self.probe_train_loader
        probe_epoch = 0
        avg_loss, avg_acc = 0.0, 0.0
        
        while (probe_epoch == 0 or avg_loss > self.min_train_loss) and (
            self.n_epochs is None or probe_epoch < self.n_epochs
        ):            
            total_loss, total_acc, n_batches = 0.0, 0.0, 0
            for batch in self.probe_train_loader:
                loss, acc = self._step(pl_module, batch, train=True)
                total_loss += float(loss)
                total_acc += float(acc)
                n_batches += 1
    
            avg_loss = total_loss / n_batches
            avg_acc = total_acc / n_batches
            print(f"Probe epoch {probe_epoch} loss:", avg_loss)
            print(f"Probe epoch {probe_epoch} acc:", avg_acc)
            probe_epoch += 1

        # Log final train metrics
        pl_module.log("online_train_loss", avg_loss, on_step=False, on_epoch=True, prog_bar=False)
        pl_module.log("online_train_acc", avg_acc, on_step=False, on_epoch=True, prog_bar=False)

        if was_training:
            pl_module.train()

    def on_validation_epoch_end(self, trainer: L.Trainer, pl_module: L.LightningModule) -> None:
        """
        Train the probe head for one pass over probe_train_loader, after SSL epoch ends.
        """
        # workaround for num_sanity_val_steps>0
        if not hasattr(self, "finetuner"):
            self._init_model_opt(trainer, pl_module)
        
        was_training = pl_module.training
        pl_module.eval()
        self.finetuner.eval()

        total_loss, total_acc, n_batches = 0.0, 0.0, 0

        with torch.no_grad():
            for batch in self.probe_val_loader:
                loss, acc = self._step(pl_module, batch, train=False)
                total_loss += float(loss)
                total_acc += float(acc)
                n_batches += 1

        avg_loss = total_loss / n_batches
        avg_acc = total_acc / n_batches

        # Log epoch metrics
        pl_module.log("online_val_loss", avg_loss, on_step=False, on_epoch=True, prog_bar=False)
        pl_module.log("online_val_acc", avg_acc, on_step=False, on_epoch=True, prog_bar=False)

        if was_training:
            pl_module.train()

In [32]:
config

ExperimentConfig(name='ms1-mz-200_peaks-sqrt-clf_run_retrain', data=DataConfig(train_dir='train_mzml', val_dir='val_mzml', batch_size=2000, max_num_peaks=200), model=ModelConfig(d_model=256, nhead=8, dim_feedforward=512, n_layers=6, dropout=0.1, n_bins=1200, bin_mz_min=300, bin_mz_max=1500, masked_peaks_fraction=0.3), optimizer=OptimizerConfig(lr=0.0005, warmup_iters=1000, cosine_schedule_period_iters=64000), training=TrainingConfig(checkpoint_path='./train_checkpoints', max_epochs=1000, gradient_clip_val=5, accelerator='gpu', devices=1))

In [33]:
model = MS1Encoder(
    d_model=config.model.d_model,
    nhead=config.model.nhead,
    dim_feedforward=config.model.dim_feedforward,
    n_layers=config.model.n_layers,
    dropout=config.model.dropout,
    n_bins=config.model.n_bins,
    bin_mz_min=config.model.bin_mz_min,
    bin_mz_max=config.model.bin_mz_max,
    masked_peaks_fraction=config.model.masked_peaks_fraction,
    lr=config.optimizer.lr,
    warmup_iters=config.optimizer.warmup_iters,
    cosine_schedule_period_iters=config.optimizer.cosine_schedule_period_iters,
)

In [34]:
root_dir = os.path.join(config.training.checkpoint_path, "foundation_model")
os.makedirs(root_dir, exist_ok=True)

logger = L.loggers.TensorBoardLogger(
    os.path.join(root_dir, "lightning_logs"),
    name=config.name,
)

In [35]:
# TODO: number of classes needs to be parametrized (based on number of classes in val subset)
# and validation classes need to be "renamed" (mapped) to always be 0 < C < n_classes
meta_df.filter(meta_df["organism"].is_in(val_organisms))["genus_class"].value_counts()

genus_class,count
i64,u32
0,42
1,27


In [36]:
retrain_finetuner = FineTuner(
    encoder_output_dim=config.model.d_model, 
    num_classes=2,#len(genus_class),
    target_key="label",
    probe_train_loader=probe_train_loader,
    probe_val_loader=probe_val_loader,
    class_weights=None,
    lr=1e-2,
    n_epochs=30,
    # min_train_loss=0.05,
)
# checkpoint_callback = L.ModelCheckpoint(every_n_epochs=100, save_top_k=-1, save_last=True)

trainer = L.Trainer(
    logger=logger,
    default_root_dir=root_dir,
    callbacks=[retrain_finetuner], #checkpoint_callback],
    accelerator="gpu", #config.training.accelerator,
    devices=config.training.devices,
    max_epochs=500, #config.training.max_epochs,
    gradient_clip_val=config.training.gradient_clip_val,
    num_sanity_val_steps=2,
    log_every_n_steps=10,
)

/home/mpominova/.local/share/mamba/envs/lcms-foundation/lib/python3.11/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/mpominova/.local/share/mamba/envs/lcms-foundat ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [ ]:
# Train the model
trainer.fit(
    model, 
    train_loader, 
    val_dataloaders=[val_loader],
    # ckpt_path=ckpt_path, # 
)


Validation DataLoader 0:  80%|███████▉  | 86/108 [01:17<00:19,  1.11it/s]

In [ ]:
# Can it be that end_validation_epoch happens before end_train_epoch? why? 

In [ ]:
# - to build learning curves (later)
    # - several experiments with multiple datasets of various sizes
    # - we can either train for a fixed number of epochs, or steps
    # (probably, steps would be better)